In [ ]:
#wdo/geometry/bbox.py
def bbox_from_feature(feature):
    """
    Return (min_lon, min_lat, max_lon, max_lat)
    for Polygon or MultiPolygon.
    """

    coords = feature["geometry"]["coordinates"]
    geom_type = feature["geometry"]["type"]

    points = []

    if geom_type == "Polygon":
        for ring in coords:
            for lon, lat in ring:
                points.append((lon, lat))

    elif geom_type == "MultiPolygon":
        for polygon in coords:
            for ring in polygon:
                for lon, lat in ring:
                    points.append((lon, lat))

    else:
        raise ValueError(f"Unsupported geometry type: {geom_type}")

    lons = [p[0] for p in points]
    lats = [p[1] for p in points]

    return min(lons), min(lats), max(lons), max(lats)

#wdo/maps/geojson_helpers.py
#from ipyleaflet import GeoJSON
#from wdo.geometry.bbox import bbox_from_feature


def add_geojson(map_obj, data, style=None):
    """Add GeoJSON layer to map."""
    
    if style is None:
        style = {
            "color": "#1f77b4",
            "fillColor": "#1f77b4",
            "weight": 2,
            "fillOpacity": 0.5,
        }

    layer = GeoJSON(data=data, style=style)
    map_obj.add_layer(layer)

    return layer


def fit_map_to_geojson(map_obj, data):
    """Fit map to bounds of GeoJSON."""

    features = data["features"]

    min_lon, min_lat = float("inf"), float("inf")
    max_lon, max_lat = float("-inf"), float("-inf")

    for feature in features:
        b = bbox_from_feature(feature)

        min_lon = min(min_lon, b[0])
        min_lat = min(min_lat, b[1])
        max_lon = max(max_lon, b[2])
        max_lat = max(max_lat, b[3])

    # Leaflet expects [lat, lon]
    bounds = [[min_lat, min_lon], [max_lat, max_lon]]

    map_obj.fit_bounds(bounds)